# 🌳 Tree of Thoughts: Propose, Evaluate and Search Intermediate Steps

## Learning Objectives
In this notebook, you will learn:
1. **Thoughts as states** - Model a problem as states and moves, with the rules enforced in code
2. **The thought generator** - Let the LLM propose several candidate next steps from a state
3. **The state evaluator** - Score how promising each state is (LLM where judgement is needed, code where the answer is exact)
4. **Beam search in LangGraph** - Keep the best few states per level in an `expand → evaluate` loop
5. **Measuring it** - Compare against single-pass Chain-of-Thought on puzzles checked in code

## Where this fits
`01_self_consistency.ipynb` sampled whole answers and voted on the final result. Tree of Thoughts (ToT) spends its extra LLM calls on **intermediate steps** instead: it branches, evaluates partial solutions and drops the unpromising ones, so a bad early step does not doom the whole answer. `03_lats.ipynb` then replaces the fixed level-by-level beam with a Monte Carlo tree-search policy.

## Prerequisites
- LangGraph state, nodes and conditional edges - `modules/03_react_agents/` and `modules/10_planning/notebooks/02_planning_decompose_execute_replan.ipynb`
- Structured output - `modules/04_langchain_agents_and_middleware/notebooks/03_structured_output.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

| Property | Value |
|---|---|
| Origin | Yao et al., *Tree of Thoughts: Deliberate Problem Solving with Large Language Models*, NeurIPS 2023 ([arXiv:2305.10601](https://arxiv.org/abs/2305.10601)) |

---
## 📦 Step 1: Environment Setup

Load the API keys from the project-root `.env`. LangSmith tracing is optional: it is used only if your `.env` turns it on.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import os
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# Optional: LangSmith traces every LLM call if the .env sets LANGSMITH_TRACING=true.
# Nothing in this notebook depends on it.
tracing = os.getenv("LANGSMITH_TRACING", os.getenv("LANGCHAIN_TRACING_V2", "false")).lower() == "true"
print(f"✅ Environment ready (LangSmith tracing: {'on' if tracing else 'off'})")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: A creative proposer and a deterministic evaluator
# ============================================================================
import threading

from aied_helpers import get_openai_llm
from langchain_core.callbacks import BaseCallbackHandler


class CallCounter(BaseCallbackHandler):
    """Counts LLM calls so we can compare the cost of CoT and ToT."""

    def __init__(self):
        self.calls = 0
        self._lock = threading.Lock()

    def on_llm_start(self, *args, **kwargs):
        with self._lock:
            self.calls += 1

    def on_chat_model_start(self, *args, **kwargs):
        with self._lock:
            self.calls += 1


counter = CallCounter()

# Proposer: temperature 0.7 so re-expanding a state can suggest different moves
proposer_llm = get_openai_llm(model="gpt-4o-mini", temperature=0.7, callbacks=[counter])
# Evaluator: temperature 0 and a token cap (a rambling evaluation is cut off, not paid for)
evaluator_llm = get_openai_llm(model="gpt-4o-mini", temperature=0, max_tokens=300, callbacks=[counter])

print(f"✅ LLMs initialized: {proposer_llm.model_name} (proposer T=0.7, evaluator T=0)")

---
## 🎲 Part 1: Thoughts as States — the Game of 24

ToT needs a problem where the path matters. The **Game of 24** (the benchmark of the ToT paper) fits: combine four numbers with `+ - * /`, each exactly once, to make 24. A step that looks fine (say `8 * 9 = 72`) can leave numbers that can no longer reach 24, and single-pass reasoning often writes an expression that does not even use the right numbers.

- **State**: the numbers still in play, plus the steps taken so far.
- **Move ("thought")**: pick two numbers and one operator; they are replaced by the result.
- **Rules in code**: a move is applied by Python with exact fractions, so an invalid or mis-computed move proposed by the LLM is simply dropped — the tree only ever contains legal states.

### Key Concepts:
- **Thought**: one intermediate step, small enough for the LLM to propose several alternatives
- **Environment**: the code that applies moves and checks the goal; the LLM never does the arithmetic of record

In [ ]:
# ============================================================================
# GAME OF 24 ENVIRONMENT: exact arithmetic, legal moves, goal checks
# ============================================================================
import ast
from collections import Counter
from fractions import Fraction
from typing import Literal, Optional

from pydantic import BaseModel, Field


def fmt(x: Fraction) -> str:
    return str(x.numerator) if x.denominator == 1 else f"{x.numerator}/{x.denominator}"


class Move(BaseModel):
    left: str = Field(description="First number, copied exactly from the current list (e.g. '8' or '8/3')")
    op: Literal["+", "-", "*", "/"]
    right: str = Field(description="Second number, copied exactly from the current list")


def apply_move(numbers: list, move: Move):
    """Return (new_numbers, step_text), or None if the move is not legal for these numbers."""
    try:
        a, b = Fraction(move.left), Fraction(move.right)
    except (ValueError, ZeroDivisionError):
        return None
    rest = list(numbers)
    for x in (a, b):
        if x not in rest:
            return None  # the LLM used a number that is not in play
        rest.remove(x)
    if move.op == "/" and b == 0:
        return None
    result = {"+": a + b, "-": a - b, "*": a * b, "/": a / b if b else None}[move.op]
    return rest + [result], f"{fmt(a)} {move.op} {fmt(b)} = {fmt(result)}"


def final_moves_to_24(numbers: list) -> list:
    """For a two-number state, every single operation that makes exactly 24 (exact check in code)."""
    a, b = numbers
    options = [(a + b, "+", a, b), (a * b, "*", a, b), (a - b, "-", a, b), (b - a, "-", b, a)]
    if b:
        options.append((a / b, "/", a, b))
    if a:
        options.append((b / a, "/", b, a))
    return [f"{fmt(x)} {op} {fmt(y)} = 24" for value, op, x, y in options if value == 24]


def check_expression(expression: str, numbers: list) -> bool:
    """True if `expression` uses exactly `numbers` with + - * / and equals 24 (used to grade CoT)."""
    expression = expression.replace("×", "*").replace("÷", "/").split("=")[0]
    used = []

    def ev(node):
        if isinstance(node, ast.BinOp):
            a, b = ev(node.left), ev(node.right)
            ops = {ast.Add: lambda: a + b, ast.Sub: lambda: a - b, ast.Mult: lambda: a * b, ast.Div: lambda: a / b}
            return ops[type(node.op)]()
        if isinstance(node, ast.Constant) and isinstance(node.value, int):
            used.append(node.value)
            return Fraction(node.value)
        raise ValueError("unsupported syntax")

    try:
        value = ev(ast.parse(expression.strip(), mode="eval").body)
    except (SyntaxError, ValueError, KeyError, ZeroDivisionError):
        return False
    return value == 24 and Counter(used) == Counter(numbers)


start = [Fraction(n) for n in (4, 7, 8, 8)]
print(apply_move(start, Move(left="8", op="/", right="8")))   # legal
print(apply_move(start, Move(left="5", op="+", right="8")))   # 5 is not in play -> None
print(final_moves_to_24([Fraction(3), Fraction(8)]))
print(check_expression("(7 - 8/8) * 4", [4, 7, 8, 8]), check_expression("4 * 7 - 8/8", [4, 7, 8, 8]))

---
## 💡 Part 2: The Thought Generator

The proposer sees one state and returns up to `k` candidate moves as structured output. It does not have to be right — the environment drops illegal moves, and the evaluator ranks the legal ones. Asking for several alternatives per state is what turns a chain into a tree.

In [ ]:
# ============================================================================
# THOUGHT GENERATOR: the LLM proposes candidate next moves
# ============================================================================
from typing import List


class Proposals(BaseModel):
    moves: List[Move] = Field(description="Different candidate next steps")


proposer = proposer_llm.with_structured_output(Proposals)


def propose(numbers: list, k: int = 8) -> list:
    """Return legal (new_numbers, step_text) pairs proposed by the LLM for this state."""
    prompt = (
        f"Game of 24. Current numbers: {[fmt(x) for x in numbers]}. "
        f"Propose up to {k} different next steps. Each step combines exactly two numbers "
        "from the list with one of + - * /."
    )
    proposals = proposer.invoke(prompt)
    legal = [r for r in (apply_move(numbers, m) for m in proposals.moves) if r]
    return legal


children = propose(start)
print(f"{len(children)} legal thoughts from {[fmt(x) for x in start]}:")
for new_numbers, step in children:
    print(f"   {step:<16} -> {[fmt(x) for x in new_numbers]}")

---
## ⚖️ Part 3: The State Evaluator

The evaluator answers *"can these numbers still reach 24?"* for every candidate state:

- **Three numbers left** — too many combinations to check by eye, so an LLM judges `sure / likely / impossible` after a few quick attempts (the value prompt of the ToT paper). Its verdict becomes a score (3 / 1 / 0).
- **Two numbers left** — there are only six operations, so **code** answers exactly. Where code can decide, it should: an LLM judging `[4, 24]` as "sure" because 24 appears in the list is a real failure we saw while prototyping.

The evaluator is the quality bottleneck of ToT: a state wrongly scored `impossible` is pruned, and no amount of search brings it back.

In [ ]:
# ============================================================================
# STATE EVALUATOR: LLM verdict for 3 numbers, exact code check for 2
# ============================================================================
from openai import LengthFinishReasonError

class Value(BaseModel):
    attempts: str = Field(description="At most three short attempts, e.g. '4*6=24 but 5 is left over'")
    verdict: Literal["sure", "likely", "impossible"]


VALUE_PROMPT = """Game of 24. Evaluate whether the numbers {numbers} can reach exactly 24.
Rules: every number in the list must be used exactly once, combined with + - * /.
A 24 already in the list does not count unless it is the only number left.
Try a few combinations, then answer:
- sure: you found a combination that gives exactly 24
- likely: a combination seems possible
- impossible: the numbers are clearly too big or too small, or every combination you tried fails"""
SCORE = {"sure": 3.0, "likely": 1.0, "impossible": 0.0}

evaluator = evaluator_llm.with_structured_output(Value)


def evaluate_states(states: list) -> list:
    """Score each number list; LLM calls for 3-number states run concurrently."""
    scores = [None] * len(states)
    llm_idx = []
    for i, numbers in enumerate(states):
        if len(numbers) == 2:
            scores[i] = 3.0 if final_moves_to_24(numbers) else 0.0
        else:
            llm_idx.append(i)
    if llm_idx:
        prompts = [VALUE_PROMPT.format(numbers=[fmt(x) for x in states[i]]) for i in llm_idx]
        # return_exceptions: an evaluation cut off by max_tokens counts as "likely" instead of crashing
        verdicts = evaluator.batch(prompts, return_exceptions=True)
        for i, v in zip(llm_idx, verdicts):
            if isinstance(v, LengthFinishReasonError):
                scores[i] = SCORE["likely"]
            elif isinstance(v, Exception):
                raise v  # anything else (auth, network) is a real error
            else:
                scores[i] = SCORE[v.verdict]
    return scores


example_states = [[Fraction(n) for n in s] for s in ([7, 8, 1], [4, 1, 16], [8, 3], [8, 4])]
for numbers, score in zip(example_states, evaluate_states(example_states)):
    print(f"   {[fmt(x) for x in numbers]!s:<14} score {score}")

---
## 🔁 Part 4: Beam Search as a LangGraph Loop

The search keeps a **frontier** of at most `BEAM` states. Each round:

1. **`expand`** asks the proposer for moves from every frontier state and collects the distinct resulting states.
2. **`evaluate`** scores them. If a two-number state can make 24, we are done. Otherwise the best `BEAM` states with a score above zero become the new frontier (**pruning** everything else).
3. A **pure router** ends the loop when a solution is found, the frontier is empty, or the two expansion rounds (4 → 3 → 2 numbers) are used up.

The LLM verdicts are written to state by `evaluate`; the router only reads state.

In [ ]:
# ============================================================================
# TOT GRAPH: expand -> evaluate -> (expand | END)
# ============================================================================
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

BEAM = 5            # states kept per level
PROPOSALS = 8       # moves requested per state
VERBOSE = True      # the experiment in Part 5 turns the trace off


class ToTState(TypedDict):
    numbers: list
    frontier: list          # [{"numbers": [...], "steps": [...]}]
    candidates: list        # expanded but not yet evaluated
    depth: int
    solution: Optional[list]


def start_node(state: ToTState) -> dict:
    root = {"numbers": [Fraction(n) for n in state["numbers"]], "steps": []}
    return {"frontier": [root], "candidates": [], "depth": 0, "solution": None}


def expand_node(state: ToTState) -> dict:
    seen, candidates = set(), []
    for node in state["frontier"]:
        for new_numbers, step in propose(node["numbers"], k=PROPOSALS):
            key = tuple(sorted(new_numbers))
            if key not in seen:  # two paths to the same numbers are one state
                seen.add(key)
                candidates.append({"numbers": new_numbers, "steps": node["steps"] + [step]})
    if VERBOSE:
        print(f"🌱 depth {state['depth']}: {len(state['frontier'])} state(s) expanded into {len(candidates)} distinct candidates")
    return {"candidates": candidates}


def evaluate_node(state: ToTState) -> dict:
    candidates = state["candidates"]
    scores = evaluate_states([c["numbers"] for c in candidates])
    for c, score in zip(candidates, scores):
        if len(c["numbers"]) == 2 and score > 0:
            solution = c["steps"] + [final_moves_to_24(c["numbers"])[0]]
            if VERBOSE:
                print(f"🎯 solution: {solution}")
            return {"solution": solution, "depth": state["depth"] + 1}
    ranked = sorted(zip(scores, range(len(candidates))), key=lambda t: -t[0])
    frontier = [candidates[i] for score, i in ranked[:BEAM] if score > 0]
    if VERBOSE:
        kept = ", ".join(f"{[fmt(x) for x in c['numbers']]}" for c in frontier)
        print(f"⚖️  kept {len(frontier)}/{len(candidates)}: {kept or '(none)'}")
    return {"frontier": frontier, "candidates": [], "depth": state["depth"] + 1}


def route(state: ToTState) -> str:
    if state["solution"] or not state["frontier"] or state["depth"] >= 2:
        return "end"
    return "expand"


builder = StateGraph(ToTState)
builder.add_node("start", start_node)
builder.add_node("expand", expand_node)
builder.add_node("evaluate", evaluate_node)
builder.add_edge(START, "start")
builder.add_edge("start", "expand")
builder.add_edge("expand", "evaluate")
builder.add_conditional_edges("evaluate", route, {"expand": "expand", "end": END})
tot_app = builder.compile()

print("🧩 Nodes:", [n for n in tot_app.get_graph().nodes if not n.startswith("__")])

In [ ]:
# ============================================================================
# RUN: one puzzle, with the trace
# ============================================================================
counter.calls = 0
result = tot_app.invoke({"numbers": [2, 3, 9, 13]})
print(f"\nSolution: {result['solution']} | LLM calls: {counter.calls}")

Read the trace: the first `expand` produced a handful of three-number states, `evaluate` kept the most promising ones, and the second round checked the two-number states exactly. A state the evaluator scored `impossible` never gets expanded, which is where ToT saves calls — and where a wrong verdict costs it the solution.

---
## 📊 Part 5: ToT vs Single-Pass Chain-of-Thought

Eight solvable puzzles. The CoT baseline asks for an expression with step-by-step reasoning (3 samples per puzzle); every expression is **checked in code** (right numbers, exactly once, equals 24). ToT runs once per puzzle. We also count LLM calls, because ToT is not free.

In [ ]:
# ============================================================================
# EXPERIMENT: CoT (3 samples per puzzle) vs ToT (1 run per puzzle)
# ============================================================================
import time

PUZZLES = [[2, 2, 5, 8], [2, 6, 8, 11], [7, 8, 9, 10], [2, 3, 9, 13],
           [11, 12, 12, 13], [5, 8, 12, 12], [4, 7, 11, 12], [1, 7, 9, 11]]


class Answer(BaseModel):
    reasoning: str
    expression: str = Field(description="One arithmetic expression using each given number exactly once, e.g. (8-4)*(7-1)")


cot = proposer_llm.with_structured_output(Answer)
COT_SAMPLES = 3

counter.calls = 0
t0 = time.time()
cot_prompts = [f"Use the numbers {p}, each exactly once, with + - * / and parentheses to make exactly 24. Think step by step."
               for p in PUZZLES for _ in range(COT_SAMPLES)]
cot_answers = cot.batch(cot_prompts)
cot_ok = [sum(check_expression(a.expression, p) for a in cot_answers[i * COT_SAMPLES:(i + 1) * COT_SAMPLES])
          for i, p in enumerate(PUZZLES)]
cot_calls, cot_time = counter.calls, time.time() - t0

VERBOSE = False
counter.calls = 0
t0 = time.time()
tot_results = tot_app.batch([{"numbers": p} for p in PUZZLES])
tot_calls, tot_time = counter.calls, time.time() - t0
VERBOSE = True

print(f"{'puzzle':<18}{'CoT correct':>12}{'ToT':>6}   ToT steps")
for p, ok, r in zip(PUZZLES, cot_ok, tot_results):
    print(f"{str(p):<18}{ok:>8}/{COT_SAMPLES}{'✅' if r['solution'] else '❌':>7}   {r['solution'] or ''}")
print(f"\nCoT: {sum(cot_ok)}/{len(PUZZLES) * COT_SAMPLES} samples correct, {cot_calls} LLM calls, {cot_time:.0f}s")
print(f"ToT: {sum(bool(r['solution']) for r in tot_results)}/{len(PUZZLES)} puzzles solved, {tot_calls} LLM calls, {tot_time:.0f}s")

In our runs, single-pass CoT solved between a twentieth and a quarter of the samples (its answers often use a number twice or skip one), while ToT solved about half of the puzzles at roughly 10–15 LLM calls per puzzle. Your exact numbers will differ — the table above is the measurement for this run. Two things hold in every run:

- **Every ToT solution is valid by construction**: the environment applied each step, so there is nothing to double-check.
- **ToT's failures are evaluator failures**: the solving state was generated but scored `impossible` and pruned, or never proposed. A wider `BEAM`, more `PROPOSALS`, or a better value prompt trades more calls for a higher solve rate.

---
## 📝 Summary

In this notebook, we learned:

### 1. Thoughts as states
- Model the problem as states and moves, and let code apply the rules: the tree only ever holds legal states.

### 2. The thought generator
- The LLM proposes several candidate steps per state; illegal proposals are simply dropped.

### 3. The state evaluator
- Use an LLM verdict where judgement is needed and an exact code check where it is not; the evaluator is ToT's bottleneck.

### 4. Beam search in LangGraph
- `expand → evaluate` in a loop, a frontier of the best `BEAM` states, and a pure router with a depth limit.

### 5. Measuring it
- Checked in code against CoT: ToT solves more puzzles, validly, at several times the LLM calls.

### Next Steps
- **`03_lats.ipynb`**: Language Agent Tree Search — spend the same kind of budget adaptively with Monte Carlo tree search, value backpropagation and reflections on failed paths